<a href="https://colab.research.google.com/github/anisha-dev26/BIS_LAB-1BF24CS054-/blob/main/GeneticAlgo.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
# ============================================================
# FEATURE SELECTION IN MACHINE LEARNING USING GENETIC ALGORITHM
# Based on: LASSO-mCGA Research Paper
# ============================================================

import numpy as np

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from sklearn.metrics import (
    roc_auc_score,
    accuracy_score,
    precision_score,
    recall_score,
    f1_score
)


# ============================================================
# 1. LOAD DATASET
# ============================================================

data = load_breast_cancer()

X = data.data
y = data.target

feature_names = data.feature_names

print("==============================================")
print(" GENETIC ALGORITHM FEATURE SELECTION")
print("==============================================")

print("Number of samples:", X.shape[0])
print("Initial number of features:", X.shape[1])


# ============================================================
# 2. TRAIN-TEST SPLIT
# ============================================================

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)


# ============================================================
# 3. FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)


# ============================================================
# 4. GENETIC ALGORITHM PARAMETERS
# ============================================================

num_features = X_train.shape[1]

population_size = 10
generations = 20

mutation_rate = 0.10
crossover_rate = 0.80


# ============================================================
# 5. FITNESS FUNCTION
# ============================================================
# Fitness = AUC-ROC using SVM
#
# The paper uses SVM and AUC-ROC to evaluate the selected
# feature subset.
# ============================================================

def fitness_function(chromosome):

    selected_features = np.where(chromosome == 1)[0]

    # No feature selected
    if len(selected_features) == 0:
        return 0

    X_train_selected = X_train[:, selected_features]
    X_test_selected = X_test[:, selected_features]

    # SVM classifier
    model = SVC(
        kernel="linear",
        probability=True,
        random_state=42
    )

    model.fit(
        X_train_selected,
        y_train
    )

    # Probability prediction
    y_probability = model.predict_proba(
        X_test_selected
    )[:, 1]

    # AUC-ROC is fitness value
    auc = roc_auc_score(
        y_test,
        y_probability
    )

    return auc


# ============================================================
# 6. INITIALIZE POPULATION
# ============================================================

population = np.random.randint(
    0,
    2,
    size=(population_size, num_features)
)


# ============================================================
# 7. GENETIC ALGORITHM
# ============================================================

best_chromosome = None
best_fitness = 0


for generation in range(1, generations + 1):

    print("\n==============================================")
    print("Generation:", generation)
    print("==============================================")

    fitness_values = []

    # --------------------------------------------------------
    # Calculate fitness for every chromosome
    # --------------------------------------------------------

    for i in range(population_size):

        chromosome = population[i]

        fitness = fitness_function(
            chromosome
        )

        fitness_values.append(fitness)

        selected_count = np.sum(chromosome)

        print(
            "Chromosome",
            i + 1,
            "| Selected Features:",
            selected_count,
            "| Fitness (AUC):",
            round(fitness, 4)
        )

    fitness_values = np.array(
        fitness_values
    )

    # --------------------------------------------------------
    # Find best chromosome
    # --------------------------------------------------------

    best_index = np.argmax(
        fitness_values
    )

    generation_best = population[
        best_index
    ].copy()

    generation_fitness = fitness_values[
        best_index
    ]

    print("\nBest Chromosome:")
    print("Chromosome:", best_index + 1)
    print(
        "Fitness:",
        round(generation_fitness, 4)
    )

    # --------------------------------------------------------
    # Save overall best solution
    # --------------------------------------------------------

    if generation_fitness > best_fitness:

        best_fitness = generation_fitness

        best_chromosome = (
            generation_best.copy()
        )

    # --------------------------------------------------------
    # SELECTION
    # --------------------------------------------------------
    # Select chromosomes using tournament selection
    # --------------------------------------------------------

    new_population = []

    for i in range(population_size):

        # Pick two random chromosomes
        a = np.random.randint(
            population_size
        )

        b = np.random.randint(
            population_size
        )

        # Select the fitter chromosome
        if fitness_values[a] > fitness_values[b]:

            winner = population[a].copy()

        else:

            winner = population[b].copy()

        new_population.append(winner)

    new_population = np.array(
        new_population
    )


    # ========================================================
    # 8. CROSSOVER
    # ========================================================

    for i in range(
        0,
        population_size - 1,
        2
    ):

        if np.random.rand() < crossover_rate:

            point = np.random.randint(
                1,
                num_features
            )

            parent1 = new_population[i].copy()
            parent2 = new_population[i + 1].copy()

            # Single-point crossover
            new_population[i][point:] = (
                parent2[point:]
            )

            new_population[i + 1][point:] = (
                parent1[point:]
            )


    # ========================================================
    # 9. MUTATION
    # ========================================================

    for i in range(population_size):

        for j in range(num_features):

            if np.random.rand() < mutation_rate:

                # Flip bit
                new_population[i][j] = (
                    1 - new_population[i][j]
                )


    # ========================================================
    # 10. ELITISM
    # ========================================================
    # Keep the best solution in the next generation
    # ========================================================

    new_population[0] = best_chromosome.copy()

    population = new_population


# ============================================================
# 11. FINAL SOLUTION
# ============================================================

print("\n\n==============================================")
print(" FINAL SOLUTION")
print("==============================================")


selected_features = np.where(
    best_chromosome == 1
)[0]


print("\nSelected Feature Indices:")

print(selected_features)


print("\nSelected Feature Names:")

for index in selected_features:

    print(
        index,
        "->",
        feature_names[index]
    )


print(
    "\nTotal Selected Features:",
    len(selected_features)
)


# ============================================================
# 12. FINAL SVM MODEL
# ============================================================

X_train_final = X_train[
    :,
    selected_features
]

X_test_final = X_test[
    :,
    selected_features
]


final_model = SVC(
    kernel="linear",
    probability=True,
    random_state=42
)


final_model.fit(
    X_train_final,
    y_train
)


# ============================================================
# 13. PREDICTION
# ============================================================

y_pred = final_model.predict(
    X_test_final
)

y_probability = final_model.predict_proba(
    X_test_final
)[:, 1]


# ============================================================
# 14. PERFORMANCE
# ============================================================

final_auc = roc_auc_score(
    y_test,
    y_probability
)

accuracy = accuracy_score(
    y_test,
    y_pred
)

precision = precision_score(
    y_test,
    y_pred
)

recall = recall_score(
    y_test,
    y_pred
)

f1 = f1_score(
    y_test,
    y_pred
)


# ============================================================
# 15. DISPLAY FINAL RESULTS
# ============================================================

print("\n==============================================")
print(" FINAL PERFORMANCE")
print("==============================================")

print(
    "Best Fitness (AUC-ROC):",
    round(best_fitness, 4)
)

print(
    "Final AUC-ROC:",
    round(final_auc, 4)
)

print(
    "Accuracy:",
    round(accuracy, 4)
)

print(
    "Precision:",
    round(precision, 4)
)

print(
    "Recall:",
    round(recall, 4)
)

print(
    "F1 Score:",
    round(f1, 4)
)

print("\n==============================================")
print(" PROGRAM COMPLETED")
print("==============================================")

 GENETIC ALGORITHM FEATURE SELECTION
Number of samples: 569
Initial number of features: 30

Generation: 1
Chromosome 1 | Selected Features: 16 | Fitness (AUC): 0.9927
Chromosome 2 | Selected Features: 18 | Fitness (AUC): 0.997
Chromosome 3 | Selected Features: 12 | Fitness (AUC): 0.998
Chromosome 4 | Selected Features: 11 | Fitness (AUC): 0.9788
Chromosome 5 | Selected Features: 13 | Fitness (AUC): 0.9917
Chromosome 6 | Selected Features: 12 | Fitness (AUC): 0.995
Chromosome 7 | Selected Features: 15 | Fitness (AUC): 0.9964
Chromosome 8 | Selected Features: 17 | Fitness (AUC): 0.9947
Chromosome 9 | Selected Features: 21 | Fitness (AUC): 0.9954
Chromosome 10 | Selected Features: 16 | Fitness (AUC): 0.9954

Best Chromosome:
Chromosome: 3
Fitness: 0.998

Generation: 2
Chromosome 1 | Selected Features: 12 | Fitness (AUC): 0.998
Chromosome 2 | Selected Features: 11 | Fitness (AUC): 0.995
Chromosome 3 | Selected Features: 17 | Fitness (AUC): 0.9924
Chromosome 4 | Selected Features: 12 | Fitn